In [2]:
import torch, re
print("Torch version:", torch.__version__)
print("Torch CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0))

Torch version: 2.8.0+cu128
Torch CUDA version: 12.8
CUDA available: True
Device name: NVIDIA GeForce RTX 3090


In [3]:
#!/usr/bin/env python
# coding: utf-8

# ===============================
# 1. Install dependencies
# ===============================
# !pip install -r requirements.txt

# ===============================
# 2. Verify GPU availability
# ===============================
import torch
import os
import subprocess
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download, login
from trl import GRPOTrainer, GRPOConfig
print("✅ CUDA available:", torch.cuda.is_available())
print("✅ Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# print the current directory
print(os.getcwd())

✅ CUDA available: True
✅ Number of GPUs: 2
GPU 0: NVIDIA GeForce RTX 3090
GPU 1: NVIDIA GeForce RTX 3090
/home/imansharifi/Documents/VirtualEnvs/ReinforcementLearning/Code/ATC_LLM


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import json, os, shutil, re, random, requests, io, sys, time
import torch
import torch.nn as nn
import numpy as np
import torch.distributed as dist
os.environ['TOKENIZERS_PARALLELISM'] = 'true'
Q_batch_size = 1
assert Q_batch_size == 1

model_path = "/data/Iman/shared_LLM_models/Qwen2.5-Math-7B"
beta = 0.04
num_pre_Q = 8
all_steps = 1000
max_prompt_length = 1024  
save_steps = 200
compute_gen_logps = True
clip_param = 0.2
# ref_server = "http://localhost:59875"

from ref_server import tensor_to_bytes, bytes_to_tensor, make_bytes_list, bytes_list_to_list

ds_config = {
    "train_micro_batch_size_per_gpu": Q_batch_size*num_pre_Q,
    "gradient_accumulation_steps": 2,
    "optimizer": {
        "type": "AdamW",
        "params": { "lr": 1e-6 }
    },
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 2,
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "contiguous_gradients": True,
        "stage3_gather_16bit_weights_on_model_save": True,
        "offload_optimizer": {"device": "cpu"}
    }
}

In [11]:
from peft import get_peft_model, LoraConfig, TaskType
tokenizer = AutoTokenizer.from_pretrained(model_path)
base_model = AutoModelForCausalLM.from_pretrained(model_path, 
        torch_dtype=torch.bfloat16, _attn_implementation="sdpa")

model = get_peft_model(base_model, LoraConfig(
    r=8, # rank
    lora_alpha=32, # alpha (scaling factor)
    lora_dropout=0.05, # dropout
    target_modules=[
        # "q_proj", "k_proj", "v_proj", 
        # "o_proj", 
        "up_proj", "down_proj", "gate_proj"
        ],
    bias="none",
    task_type=TaskType.CAUSAL_LM
))
gen_model = model

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [12]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params} / {total_params} ({100 * trainable_params / total_params:.4f}%)")

Trainable parameters: 15138816 / 7630755328 (0.1984%)


In [17]:
from datasets import load_dataset
dataset = load_dataset("openai/gsm8k", "main", split="train")
QAs = [{'Q':x, 'A':y.split('####')[-1].strip()} for x,y in zip(dataset['question'], dataset['answer'])]

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [18]:
from datasets import load_dataset

dataset_dir = "/data/Iman/datasets/"
# split the dataset into train and test sets
dataset = load_dataset("json", data_files=dataset_dir+"ownship_prompts_with_action_rewards_dataNov17.jsonl", split='train')

In [19]:
import re

def to_grpo_row(ex):
    prompt = (
        f"### System:\n{ex['system_prompt'].strip()}\n"
        f"### User:\n{ex['user_prompt'].strip()}. Always start your response with: {"The recommended action is:"} \n"
        f"### Assistant: "
    )
    prompt_response = ex['prompt_response']
    action = re.search(r"\b(Accelerate|Hold|Decelerate)\b", prompt_response, flags=re.IGNORECASE)
    action = action.group(1).capitalize() if action else None
    solution = "The recommended action is: " + action + "."

    return {
        "prompt": [{'content': prompt, 'role': 'user' }],
        "solution": solution,       # dict per prompt
    }

ds_train = dataset.map(to_grpo_row, remove_columns=dataset.column_names)
# ds_test = dataset["test"].map(to_grpo_row, remove_columns=ds["test"].column_names)

In [20]:
QAs = []
for sample in ds_train:
    QAs.append({'Q': sample['prompt'][0]['content'], 'A': sample['solution']})
QAs[0]

{'Q': '### System:\nYou are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, \n    such as merging or intersections points, where other agents (intruders) are approaching as well. Based on the information of the intruders, \n    the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, \n    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by \n    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.\n    Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],\n    v: ownship speed(m/s), \n    v_min: ownship minimum speed(m/s), \n    v_max: ownship maximum speed(m/s),\n    

In [156]:
from transformers import GenerationConfig
generation_config = GenerationConfig(
            max_new_tokens=9,
            do_sample=True, temperature=0.9, 
            num_return_sequences=num_pre_Q,
            pad_token_id=tokenizer.pad_token_id,
        )

In [157]:
def gen_answers(prompts):
    tip_text = []
    system_prompt = """You are a helpful assistant. A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the user with the answer.\
The reasoning process and answer are enclosed within <think> </think> and<answer> </answer> tags, respectively, i.e., <think> reasoning process here </think><answer> answer here </answer>."""

    for x in prompts:
        # print("Generating for prompt:", x)
        tip_text.append(tokenizer.apply_chat_template([
             {"role": "system", "content": system_prompt},
             {"role": "user", "content": x}], tokenize=False, add_generation_prompt=True))
    tip_inputs = tokenizer(tip_text, return_tensors="pt", padding=True, padding_side="left", add_special_tokens=False)
    prompt_length = tip_inputs["input_ids"].shape[-1]
    # if prompt_length > max_prompt_length: return []
    
    # Move inputs to the correct device
    device = gen_model.device if hasattr(gen_model, 'device') else next(gen_model.parameters()).device
    tip_inputs = {k: v.to(device) for k, v in tip_inputs.items()}
    
    with torch.inference_mode():
        tip_completion_ids = gen_model.generate(**tip_inputs, generation_config=generation_config)
    completion_ids = tip_completion_ids[:, prompt_length:]
    answers = [tokenizer.decode(x).replace('<|endoftext|>', '') for x in completion_ids]
    return answers

In [158]:
from math_verify import parse, verify, ExprExtractionConfig
def reward_correct(item, answer):
    pattern = r'\d+\.\d+|\d+/\d+|\d+'
    nums = re.findall(pattern, answer) # 使用正则表达式在answer中查找所有数字
    if len(nums) == 0: return -1.0
    lastnum = nums[-1] # 用answer中最后一个数字和ground_truth做比较
    ans = parse(lastnum, extraction_config=[ExprExtractionConfig()])
    ground_truth = parse(item["A"], extraction_config=[ExprExtractionConfig()])
    return 1 if verify(ans, ground_truth) else -1
def reward_format(item, answer):
    # pattern = r"^<think>(?:(?!</?think>)[\s\S]*?)</think>\s*<answer>(?:(?!</?answer>)[\s\S]*?)</answer><\|im_end\|>$"
    pattern = r"^<think>.*?</think><answer>.*?</answer>$"
    return 1.25 if re.match(pattern, answer, re.DOTALL | re.VERBOSE) else -1

In [159]:
def gen_samples(inputs):
    prompts = [x["Q"] for x in inputs]
    # print("Generating answers for prompts...")
    answers = gen_answers(prompts)
    # print("answers:", answers)
    if len(answers) == 0: return None, None, None, None
    rewards = []
    for i, inp in enumerate(inputs):
        for a in answers[i*num_pre_Q:(i+1)*num_pre_Q]:
            rewards.append(reward_correct(inp, a) + reward_format(inp, a))
    prompts_text = [tokenizer.apply_chat_template([
             {"role": "system", "content": system_prompt},
             {"role": "user", "content": x}], tokenize=False, add_generation_prompt=True) for x in prompts]
    prompt_inputs = tokenizer(prompts_text, return_tensors="pt", padding=True, padding_side="left", add_special_tokens=False)["input_ids"]
    output_ids = tokenizer(answers, return_tensors="pt", padding=True, padding_side="right", add_special_tokens=False)["input_ids"]
    return prompt_inputs, output_ids, torch.tensor(rewards, dtype=torch.float32), answers

In [170]:
def generate_mode(num=10, rank=0):
    # if rank == 0: print('enter generate mode')
    tic = time.time()
    xdata_batch = []
    data = {}
    for ii in range(num):
        inputs = random.sample(QAs, Q_batch_size)
        # print("Inputs:", inputs)
        prompt_inputs, output_ids, rewards, answers = gen_samples(inputs)
        # print(prompt_inputs, output_ids, rewards, answers)
        if prompt_inputs is None: continue
        if rank == 0: 
            print('rewards:', rewards)
            if ii == 5: print('answers:', answers[0])
        if (rewards.max() - rewards.min()).item() < 0.01: continue
        rewards = (rewards - rewards.mean()) / (rewards.std() + 1e-4)
        rep = output_ids.shape[0] // prompt_inputs.shape[0]
        prompt_length = prompt_inputs.shape[1]
        print("prompt_length:", prompt_length)
        Qrep = prompt_inputs.repeat(1, rep).view(-1, prompt_length)
        merged_ids = torch.cat([Qrep, output_ids], dim=1)
        data['plen'] = prompt_length
        data['rewards'] = tensor_to_bytes(rewards)
        data['inputs'] = tensor_to_bytes(merged_ids)
        data['refs'] = answers
        data['outputs'] = tensor_to_bytes(output_ids)

        if compute_gen_logps:
            # Reduntant calculations? For modifiability!
            with torch.inference_mode():
                device = gen_model.device if hasattr(gen_model, 'device') else torch.device('cuda')
                mids = merged_ids.to(device)
                if hasattr(gen_model, 'module'):
                    out = gen_model.module(mids)
                else:
                    out = gen_model(mids)
                gen_logps = get_per_token_logps(out.logits[:, :-1, :], mids[:, 1:])

            
            data['gen_logps'] = tensor_to_bytes(gen_logps[:,prompt_length-1:].cpu())

        xdata_batch.append(data)
        # requests.post(f"{ref_server}/upload", data=xdata)
    # if rank == 0: print('exit generate mode')
    # print(f'{rank}: {time.time()-tic:.3f}s')

    return xdata_batch

In [161]:
def get_per_token_logps(logits, input_ids):
    per_token_logps = [] # Use a loop to reduce memory peak.
    for logits_row, input_ids_row in zip(logits, input_ids):
        log_probs = logits_row.log_softmax(dim=-1)
        token_log_prob = torch.gather(log_probs, dim=1, index=input_ids_row.unsqueeze(1)).squeeze(1)
        per_token_logps.append(token_log_prob)
    return torch.stack(per_token_logps)

In [165]:
def GRPO_step(batch):
    prompt_length = batch['plen']
    print("prompt_length in GRPO_step:", prompt_length)
    inputs = batch['inputs'].to(engine.device)
    advantages = batch['rewards'].to(engine.device).unsqueeze(1)   # normalized in generation

    logits = engine(inputs).logits
    logits = logits[:, :-1, :]  # (B, L-1, V), exclude the last logit: it corresponds to the next token pred
    input_ids = inputs[:, 1:]  # (B, L-1), exclude the first input ID since we don't have logits for it
    
    per_token_logps = get_per_token_logps(logits, input_ids)
    per_token_logps = per_token_logps[:,prompt_length-1:]
    ref_per_token_logps = batch['refs'].to(per_token_logps.device)

    per_token_kl = torch.exp(ref_per_token_logps - per_token_logps) - (ref_per_token_logps - per_token_logps) - 1
    completion_mask = (inputs[:, prompt_length:] != tokenizer.pad_token_id).int()

    if 'gen_logps' in batch:
        ratio = torch.exp(per_token_logps - batch['gen_logps'].to(engine.device))
        clipped_ratio = torch.clamp(ratio, 1-clip_param, 1+clip_param)
        per_token_loss = torch.min(ratio * advantages, clipped_ratio * advantages)
    else: 
        per_token_loss = torch.exp(per_token_logps - per_token_logps.detach()) * advantages
        assert compute_gen_logps is False

    per_token_loss = -(per_token_loss - beta * per_token_kl)
    loss = ((per_token_loss * completion_mask).sum(dim=1) / completion_mask.sum(dim=1)).mean()
    return loss

In [163]:
def get_batch():
    return generate_mode(num=10, rank=0)

In [39]:
import torch
assert torch.cuda.is_available(), "CUDA not available"
print("CUDA devices:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

CUDA devices: 2 ['NVIDIA GeForce RTX 3090', 'NVIDIA GeForce RTX 3090']


In [169]:
# if 'genonly' in sys.argv:
# For generation-only mode, you would initialize without deepspeed
# model.to('cuda')
# generate_mode(999999)
# sys.exit()


# initialize deepspeed (keep your existing ds_config and args)
import deepspeed
engine, optimizer, _, _ = deepspeed.initialize(config=ds_config, model=model, model_parameters=model.parameters())

# Use engine as the active model object for inference/training
gen_model = engine   # for generate() calls, some setups support engine.generate()
model = engine.module   # keep a reference to the actual module

# set explicit device using engine.local_rank
device = torch.device(f"cuda:{engine.local_rank}")
torch.cuda.set_device(device)
engine.device = device
model.to(device)

# Verify device placement
print("Engine device:", engine.device, "local_rank:", getattr(engine, "local_rank", None))
print("Model device (sample param):", next(model.parameters()).device)

#from kernel.ce_kernel import fast_log_softmax_gather
#get_per_token_logps = fast_log_softmax_gather

# xdata = generate_mode(rank=torch.distributed.get_rank())
# print(xdata)

from tqdm import tqdm
progress = range(1, all_steps+1)
if torch.distributed.get_rank() == 0: progress = tqdm(progress)
for step in progress:
    print("step:", step)
    batch = get_batch()
    # while batch is None:
    #     generate_mode(rank=torch.distributed.get_rank())
    #     batch = get_batch()

    print(batch)

    loss = GRPO_step(batch)
    engine.backward(loss)
    engine.step()

    print(f"Step {step}, Loss: {loss.item():.6f}")

    if torch.distributed.get_rank() == 0:
        progress.set_description(f"Loss: {loss.item():.6f}")

    if step % save_steps == 0:
        dist.barrier()
        if torch.distributed.get_rank() == 0:
            print('saving model')
            save_name = f"./step_{step}"
            state_dict = engine.module.state_dict()
            state_dict = type(state_dict)({k: v.cpu() for k, v in state_dict.items()})
            engine.module.save_pretrained(save_name, state_dict=state_dict)
            tokenizer.save_pretrained(save_name)
        dist.barrier()

Engine device: cuda:0 local_rank: 0
Model device (sample param): cuda:0


  0%|          | 0/1000 [00:00<?, ?it/s]

step: 1
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
answers: ### System:
You are a airspace tactical de
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
answers: ### System:
You are a airspace tactical de
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
rewards: tensor([-

  0%|          | 0/1000 [00:16<?, ?it/s]

rewards: tensor([-2., -2., -2., -2., -2., -2., -2., -2.])
[]


TypeError: list indices must be integers or slices, not str